In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split,cross_val_score,GridSearchCV
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,f1_score
import joblib

In [2]:
df=pd.read_csv('../data/bank_customer_data.csv')

In [3]:
df.head()

,customer_id,age,gender,marital_status,education,employment_type,years_employed,annual_income,monthly_expenses,num_dependents,...,existing_loans,missed_payments,savings_balance,investment_flag,loan_amount,loan_tenure,interest_rate,default,churn,credit_risk
0,CUST00001,56,Female,Married,Graduate,Salaried,33,627221,27346,0,...,1,1,292746,1,708007.0,21,18.45,0,0,Medium
1,CUST00002,69,Male,Single,Graduate,Salaried,40,668372,34942,2,...,2,0,137967,0,788237.0,21,15.79,0,0,Medium
2,CUST00003,46,Male,Single,High School,Self-Employed,26,554249,26827,3,...,1,1,60548,0,1060949.0,29,22.06,1,1,High
3,CUST00004,32,Female,Single,Graduate,Self-Employed,7,593807,34816,0,...,5,1,289507,0,1103022.0,4,20.94,0,1,Medium
4,CUST00005,60,Male,Married,High School,Salaried,31,396889,13767,5,...,0,0,155872,1,938617.0,7,19.68,0,0,Medium


In [4]:
df['credit_risk']=df['credit_risk'].map({
    'Low':0,
    'Medium':1,
    'High':2
})


In [5]:
df['credit_risk'].unique()

array([1, 2, 0])

In [6]:
df.head()

,customer_id,age,gender,marital_status,education,employment_type,years_employed,annual_income,monthly_expenses,num_dependents,...,existing_loans,missed_payments,savings_balance,investment_flag,loan_amount,loan_tenure,interest_rate,default,churn,credit_risk
0,CUST00001,56,Female,Married,Graduate,Salaried,33,627221,27346,0,...,1,1,292746,1,708007.0,21,18.45,0,0,1
1,CUST00002,69,Male,Single,Graduate,Salaried,40,668372,34942,2,...,2,0,137967,0,788237.0,21,15.79,0,0,1
2,CUST00003,46,Male,Single,High School,Self-Employed,26,554249,26827,3,...,1,1,60548,0,1060949.0,29,22.06,1,1,2
3,CUST00004,32,Female,Single,Graduate,Self-Employed,7,593807,34816,0,...,5,1,289507,0,1103022.0,4,20.94,0,1,1
4,CUST00005,60,Male,Married,High School,Salaried,31,396889,13767,5,...,0,0,155872,1,938617.0,7,19.68,0,0,1


In [7]:
df.drop(['customer_id','gender','num_dependents','marital_status','loan_tenure'],axis=1,inplace=True)

In [8]:
df.columns

Index(['age', 'education', 'employment_type', 'years_employed',
       'annual_income', 'monthly_expenses', 'credit_score', 'existing_loans',
       'missed_payments', 'savings_balance', 'investment_flag', 'loan_amount',
       'interest_rate', 'default', 'churn', 'credit_risk'],
      dtype='str')

In [9]:
df['education']=df['education'].map({
    'Graduate':1,
    'High School':0,
    'Postgraduate':2
})

In [10]:
df=pd.get_dummies(df,columns=['employment_type'],drop_first=True)

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 17 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   age                            50000 non-null  int64  
 1   education                      50000 non-null  int64  
 2   years_employed                 50000 non-null  int64  
 3   annual_income                  50000 non-null  int64  
 4   monthly_expenses               50000 non-null  int64  
 5   credit_score                   50000 non-null  int64  
 6   existing_loans                 50000 non-null  int64  
 7   missed_payments                50000 non-null  int64  
 8   savings_balance                50000 non-null  int64  
 9   investment_flag                50000 non-null  int64  
 10  loan_amount                    50000 non-null  float64
 11  interest_rate                  50000 non-null  float64
 12  default                        50000 non-null  int64  
 1

In [12]:
X=df.drop(['churn','default','loan_amount','credit_risk'],axis=1)
y=df['credit_risk']

In [13]:
df['credit_risk'].value_counts(normalize=True)

credit_risk
1    0.49124
2    0.33242
0    0.17634
Name: proportion, dtype: float64

In [14]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42,stratify=y)

In [15]:
#imb_ratio=df['default'].value_counts()[0]/df['default'].value_counts()[1]
#print(imb_ratio)
models={
    'LogisticRegression':Pipeline([
        ('scaler',StandardScaler()),
        ('model',LogisticRegression(max_iter=1000,class_weight='balanced'))
    ]),
    'SVM':Pipeline([
        ('scaler',StandardScaler()),
        ('model',SVC(class_weight='balanced'))
    ]),
    'KNN':Pipeline([
        ('scaler',StandardScaler()),
        ('model',KNeighborsClassifier())
    ]),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced'),
    'Random Forest':RandomForestClassifier(class_weight='balanced'),
    'XGboost':XGBClassifier(eval_metric='mlogloss')
}

for name,model in models.items():
    model.fit(X_train,y_train)
    y_train_pred=model.predict(X_train)
    y_pred=model.predict(X_test)
    print(f" the training accuracy of the model {name} is {accuracy_score(y_train,y_train_pred)} and testing accuracy is {accuracy_score(y_test,y_pred)}")

 the training accuracy of the model LogisticRegression is 0.8819428571428571 and testing accuracy is 0.8836
 the training accuracy of the model SVM is 0.8966857142857143 and testing accuracy is 0.8946666666666667
 the training accuracy of the model KNN is 0.8964571428571428 and testing accuracy is 0.8514
 the training accuracy of the model Decision Tree is 1.0 and testing accuracy is 0.8754
 the training accuracy of the model Random Forest is 0.9999714285714286 and testing accuracy is 0.9207333333333333
 the training accuracy of the model XGboost is 0.9503714285714285 and testing accuracy is 0.9234


In [16]:
for name in ['Random Forest', 'XGboost','Decision Tree']:
    y_pred = models[name].predict(X_test)
    print(f"\n{name}:")
    print(classification_report(y_test, y_pred, target_names=['Low', 'Medium', 'High']))


Random Forest:
              precision    recall  f1-score   support

         Low       0.93      0.98      0.96      2645
      Medium       0.89      0.96      0.92      7369
        High       0.97      0.84      0.90      4986

    accuracy                           0.92     15000
   macro avg       0.93      0.92      0.93     15000
weighted avg       0.92      0.92      0.92     15000


XGboost:
              precision    recall  f1-score   support

         Low       0.93      0.99      0.96      2645
      Medium       0.90      0.95      0.92      7369
        High       0.97      0.84      0.90      4986

    accuracy                           0.92     15000
   macro avg       0.93      0.93      0.93     15000
weighted avg       0.93      0.92      0.92     15000


Decision Tree:
              precision    recall  f1-score   support

         Low       0.93      0.91      0.92      2645
      Medium       0.87      0.87      0.87      7369
        High       0.85      0.87

In [17]:
for name in ['Random Forest','Decision Tree','XGboost']:
    cv=cross_val_score(models[name],X_train,y_train,cv=5,scoring='f1_macro')
    print(f"{name}",cv.mean(),cv.std())

Random Forest 0.919927550260282 0.0046237870463060994
Decision Tree 0.8783776212387157 0.00363147286301682
XGboost 0.9231529518759617 0.0055319875674441155


In [18]:
params_xgb={
    'max_depth':[3,5,7],
    'n_estimators':[5,8,10,50,100,200],
}
Grid_model=GridSearchCV(estimator=XGBClassifier(eval_metric='logloss'),param_grid=params_xgb,n_jobs=-1,cv=5,scoring='f1_macro')
Grid_model.fit(X_train,y_train)
print(Grid_model.best_params_)
print(Grid_model.best_score_)
y_pred=Grid_model.predict(X_test)
print(accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred, target_names=['Low', 'Medium', 'High']))

{'max_depth': 7, 'n_estimators': 8}
0.9278467462508029
0.9293333333333333
              precision    recall  f1-score   support

         Low       0.93      1.00      0.96      2645
      Medium       0.90      0.97      0.93      7369
        High       0.99      0.84      0.91      4986

    accuracy                           0.93     15000
   macro avg       0.94      0.93      0.93     15000
weighted avg       0.93      0.93      0.93     15000



In [19]:
Grid_model.best_estimator_.feature_importances_

array([0.00358713, 0.00503688, 0.00488633, 0.03180483, 0.00451372,
       0.20640399, 0.09472938, 0.40251887, 0.00500475, 0.0025896 ,
       0.00393079, 0.00441441, 0.23057926], dtype=float32)

In [20]:
imp_features=pd.Series(Grid_model.best_estimator_.feature_importances_,index=X.columns).sort_values(ascending=False)

In [21]:
imp_features

missed_payments                  0.402519
employment_type_Unemployed       0.230579
credit_score                     0.206404
existing_loans                   0.094729
annual_income                    0.031805
education                        0.005037
savings_balance                  0.005005
years_employed                   0.004886
monthly_expenses                 0.004514
employment_type_Self-Employed    0.004414
interest_rate                    0.003931
age                              0.003587
investment_flag                  0.002590
dtype: float32

In [22]:
joblib.dump(Grid_model.best_estimator_, '../models/credit_risk_model.pkl')
print("Credit risk model saved.")

Credit risk model saved.
